## Data Analytics Module 3 - Delta Lake

### Loading CSV file into Unity Catalog Volume

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [ ]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["products.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/Databricks-Job-Ready-Projects/refs/heads/main/Databricks-Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading Data into a Dataframe

In [ ]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/products.csv', format='csv', header=True)
display(df.limit(10))

### Load the data into a Delta Table

#### Storing in Unity Catalog Volume

In [ ]:
delta_table_path = f"/Volumes/{catalog_name}/default/spark_lab/delta/products-delta" 
df.write.format("delta").save(delta_table_path)

#### Manipulating the Delta Table by creating a DeltaTable Object

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# Create a deltaTable object
deltaTable = DeltaTable.forPath(spark, delta_table_path)
# Update the table (reduce price of product 771 by 10%)
deltaTable.update(
   condition = "ProductID == 771",
   set = { "ListPrice": "ListPrice * 0.9" })
# View the updated data as a dataframe
deltaTable.toDF().show(10)

### Creating a dataframe from the delta dataset

In [ ]:
new_df = spark.read.format("delta").load(delta_table_path)
new_df.show(10)

### Explore Logging for the delta table

In [ ]:
deltaTable.history(10).show(10, False, True)

### Create a Data Catalog Table

In [ ]:
df.write.format("delta").saveAsTable("default.ProductsManaged")

### Accessing the Data Catalog Table

In [ ]:
%sql
USE default;
SELECT * FROM ProductsManaged;